# tui

> Terminal inspector: a tight, slick gruvbox TUI over the live namespace

In [1]:
#| default_exp tui

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
from dataclasses import dataclass
from paar.core import VarInfo
from paar.snapshot import PROFILES, profile_view, expand as _ns_expand, grid_page as _ns_grid

# gruvbox dark (medium) — one place to retint the whole surface
_GB = dict(bg0='#282828', bg0h='#1d2021', bg1='#3c3836', bg2='#504945', bg3='#665c54',
           bg4='#7c6f64', fg='#ebdbb2', fg2='#d5c4a1', fg4='#a89984', gray='#928374',
           red='#fb4934', green='#b8bb26', yellow='#fabd2f', blue='#83a598',
           purple='#d3869b', aqua='#8ec07c', orange='#fe8019')

def _style_dict():
    "Map the gruvbox palette onto the semantic classes the renderer emits."
    g = _GB
    return {
        'name':       g['aqua'],
        'eq':         g['gray'],
        'type':       g['bg4'],
        'val':        g['fg'],
        'val.str':    g['green'],
        'val.num':    g['purple'],
        'val.bool':   g['orange'],
        'val.none':   f"{g['gray']} italic",
        'dtype':      g['yellow'],
        'toggle':     g['orange'],
        'guide':      g['bg2'],
        'group':      f"{g['yellow']} bold",
        'more':       f"{g['blue']} italic",
        'grid.head':  f"{g['yellow']} bold",
        'grid.index': g['blue'],
        'grid.cell':  g['fg2'],
        'error':      g['red'],
        'cursor':     f"bg:{g['bg2']}",
        'bar':        g['orange'],
        'header':     f"bg:{g['bg0h']} {g['orange']} bold",
        'header.dim': f"bg:{g['bg0h']} {g['gray']}",
        'footer':     f"bg:{g['bg0h']} {g['gray']}",
        'footer.key': f"bg:{g['bg0h']} {g['aqua']}",
        'path':       f"bg:{g['bg0h']} {g['green']}",
    }

def GRUVBOX():
    "prompt_toolkit Style for the inspector (imported lazily so paar has no hard TUI dep)."
    from prompt_toolkit.styles import Style
    return Style.from_dict(_style_dict())

In [4]:
#| export
class NsSource:
    "Frontend-agnostic view/expand/grid over a plain namespace dict (mirrors bridge.Bridge)."
    def __init__(self, ns, hidden=frozenset()): self.ns, self.hidden = ns, hidden
    def view(self, profile='standard'): return profile_view(self.ns, self.hidden, profile)
    def expand(self, accessor, offset=0): return _ns_expand(self.ns, accessor, offset)
    def grid(self, accessor, roff=0, coff=0, rows=50, cols=50):
        return _ns_grid(self.ns, accessor, roff, coff, rows, cols)

In [5]:
#| export
@dataclass
class Row:
    "One rendered line: its depth, kind, and fully-styled fragments (invariant: one Row == one screen line)."
    depth:int
    kind:str                 # group | node | more | gridline
    key:tuple                # unique, hashable identity for cursor/expanded tracking
    frags:list               # list[(style, text)] with no trailing newline
    v:object=None            # VarInfo (node/more) or label str (group)
    owner:tuple=None         # gridable accessor a gridline belongs to (for paging)

def _valclass(t):
    "Colour a scalar value by its python type name."
    if t == 'str': return 'class:val.str'
    if t in ('int', 'float', 'complex'): return 'class:val.num'
    if t == 'bool': return 'class:val.bool'
    if t == 'NoneType': return 'class:val.none'
    return 'class:val'

def _cut(s, w):
    "Truncate s to width w with an ellipsis."
    s = str(s)
    return s if len(s) <= w else s[:max(0, w-1)] + '…'

In [6]:
#| export
class InspectorState:
    "Holds the expand/grid/cursor state and flattens the live namespace into a Row list."
    def __init__(self, src, profile='standard'):
        self.src = src
        self.profile = profile
        self.expanded = set()      # keys (accessor tuple or ('group',label)) currently open
        self.grids = {}            # gridable accessor -> [roff, coff]
        self.loaded = {}           # accessor -> accumulated child VarInfo (incl. trailing sentinel)
        self.cursor = 0
        self.rows = []
        self.refresh()

    # -- data --
    def _children(self, key):
        if key not in self.loaded:
            self.loaded[key] = list(self.src.expand(key, 0))
        return self.loaded[key]

    def load_more(self, key, offset):
        more = list(self.src.expand(key, offset))
        lst = list(self.loaded.get(key, []))
        if lst and getattr(lst[-1], 'more_offset', None) is not None: lst = lst[:-1]
        self.loaded[key] = lst + more

    # -- flatten --
    def refresh(self):
        "Rebuild the flat Row list from the source, honouring expand/grid state; keeps the cursor in range."
        rows = []
        for label, vs in self.src.view(self.profile):
            if label is None:
                for v in vs: self._emit(v, 0, rows)
            else:
                gkey = ('group', label)
                rows.append(self._group_row(label, vs, gkey))
                if gkey in self.expanded:
                    for v in vs: self._emit(v, 1, rows)
        self.rows = rows
        if self.cursor >= len(rows): self.cursor = max(0, len(rows)-1)

    def _emit(self, v, depth, rows):
        key = v.accessor
        rows.append(self._node_row(v, depth, key))
        if v.is_grid and key in self.grids:
            for line in self._grid_lines(v, depth+1, key): rows.append(line)
        if v.is_container and key in self.expanded:
            for c in self._children(key):
                if getattr(c, 'more_offset', None) is not None:
                    rows.append(self._more_row(c, depth+1))
                else:
                    self._emit(c, depth+1, rows)

    # -- row builders (fasthtml-style small composables) --
    def _indent(self, depth): return [('class:guide', '  ' * depth)]

    def _toggle(self, open_):
        return ('class:toggle', '▾ ' if open_ else '▸ ')

    def _node_row(self, v, depth, key):
        f = self._indent(depth)
        openable = v.is_container or v.is_grid
        f.append(self._toggle(key in self.expanded or key in self.grids) if openable
                 else ('class:guide', '  '))
        if v.is_error:
            f += [('class:name', v.name), ('class:eq', ' = '), ('class:error', v.value)]
            return Row(depth, 'node', key, f, v=v)
        f.append(('class:name', v.name))
        f.append(('class:eq', ' = '))
        typ = f'{{{v.type}: {v.shape}}}' if v.shape else f'{{{v.type}}}'
        f.append(('class:type', typ + ' '))
        f.append((_valclass(v.type), _cut(v.value, 120)))
        if v.dtype: f.append(('class:dtype', f'  {v.dtype}'))
        if v.is_grid: f.append(('class:more', '   ▦'))
        return Row(depth, 'node', key, f, v=v)

    def _group_row(self, label, vs, gkey):
        f = self._indent(0)
        f.append(self._toggle(gkey in self.expanded))
        f.append(('class:group', f'{label} ({len(vs)})'))
        return Row(0, 'group', gkey, f, v=label)

    def _more_row(self, v, depth):
        f = self._indent(depth)
        f.append(('class:guide', '  '))
        f.append(('class:more', f'… {v.value}'))
        return Row(depth, 'more', (v.accessor, v.more_offset), f, v=v)

    def _grid_lines(self, v, depth, key):
        roff, coff = self.grids[key]
        page = self.src.grid(key, roff, coff)
        pad = ('class:guide', '  ' * depth + '  ')
        if page is None:
            return [Row(depth, 'gridline', ('grid', key, 'x'), [pad, ('class:error', 'not gridable')], owner=key)]
        headers, index, cells = page['headers'], page['index'], page['cells']
        colw = [len(h) for h in headers]
        for row in cells:
            for j, c in enumerate(row): colw[j] = max(colw[j], len(c))
        colw = [min(w, 22) for w in colw]
        idxw = min(max([len(i) for i in index] + [1]), 12)
        out = []
        head = [pad, ('class:guide', ' ' * idxw + ' ')]
        for h, w in zip(headers, colw): head.append(('class:grid.head', _cut(h, w).ljust(w) + '  '))
        out.append(Row(depth, 'gridline', ('grid', key, 'h'), head, owner=key))
        for r, (ix, row) in enumerate(zip(index, cells)):
            rf = [pad, ('class:grid.index', _cut(ix, idxw).ljust(idxw) + ' ')]
            for c, w in zip(row, colw): rf.append(('class:grid.cell', _cut(c, w).ljust(w) + '  '))
            out.append(Row(depth, 'gridline', ('grid', key, r), rf, owner=key))
        npr, npc = len(index), len(headers)
        nav = (f'rows {roff}-{roff+npr}/{page["nrows"]}  '
               f'cols {coff}-{coff+npc}/{page["ncols"]}   [ ] rows   {{ }} cols')
        out.append(Row(depth, 'gridline', ('grid', key, 'nav'), [pad, ('class:more', nav)], owner=key,
                       v=page))
        return out

    # -- actions --
    @property
    def cur(self): return self.rows[self.cursor] if self.rows else None

    def move(self, d):
        if self.rows: self.cursor = max(0, min(len(self.rows)-1, self.cursor + d))

    def top(self): self.cursor = 0
    def bottom(self): self.cursor = max(0, len(self.rows)-1)

    def toggle(self):
        "Enter/→: open a group or container; toggle a lone grid; page-in a load-more sentinel."
        r = self.cur
        if r is None: return
        if r.kind == 'group':
            self.expanded ^= {r.key}
        elif r.kind == 'more':
            self.load_more(r.v.accessor, r.v.more_offset)
        elif r.kind == 'node':
            v = r.v
            if v.is_container:
                self.expanded ^= {r.key}
            elif v.is_grid:
                self.toggle_grid()
        self.refresh()

    def collapse(self):
        "←/h: close an open node, else jump to the parent row."
        r = self.cur
        if r is None: return
        if r.key in self.expanded:
            self.expanded.discard(r.key); self.refresh(); return
        d = r.depth
        for i in range(self.cursor-1, -1, -1):
            if self.rows[i].depth < d: self.cursor = i; break

    def toggle_grid(self):
        "v: show/hide the paged table for the gridable node under the cursor."
        r = self.cur
        if r is None: return
        key = r.owner if r.kind == 'gridline' else (r.key if r.kind == 'node' and r.v.is_grid else None)
        if key is None: return
        if key in self.grids: del self.grids[key]
        else: self.grids[key] = [0, 0]
        self.refresh()

    def _grid_key(self):
        r = self.cur
        if r is None: return None
        if r.kind == 'gridline': return r.owner
        if r.kind == 'node' and r.v.is_grid and r.key in self.grids: return r.key
        return None

    def page_grid(self, drow=0, dcol=0):
        "[ ]/{ }: page the active grid by one window in rows/cols, clamped to its shape."
        key = self._grid_key()
        if key is None: return
        page = self.src.grid(key, *self.grids[key])
        if page is None: return
        roff, coff = self.grids[key]
        npr, npc = len(page['index']), len(page['headers'])
        if drow: roff = max(0, min(page['nrows']-1, roff + drow*npr))
        if dcol: coff = max(0, min(page['ncols']-1, coff + dcol*npc))
        self.grids[key] = [roff, coff]
        self.refresh()

    def cycle_profile(self, d=1):
        ps = list(PROFILES)
        self.profile = ps[(ps.index(self.profile) + d) % len(ps)]
        self.refresh()

    def hard_refresh(self):
        "r: drop caches and re-read the namespace from scratch (values may have changed)."
        self.loaded.clear()
        self.refresh()

In [7]:
#| export
_FOOTER = [('class:footer.key', ' j/k'), ('class:footer', ' move  '),
           ('class:footer.key', 'l/⏎'), ('class:footer', ' open  '),
           ('class:footer.key', 'h'), ('class:footer', ' close  '),
           ('class:footer.key', 'v'), ('class:footer', ' grid  '),
           ('class:footer.key', '[ ] { }'), ('class:footer', ' page  '),
           ('class:footer.key', 'p'), ('class:footer', ' profile  '),
           ('class:footer.key', 'r'), ('class:footer', ' reload  '),
           ('class:footer.key', 'q'), ('class:footer', ' quit ')]

class InspectorTUI:
    "prompt_toolkit full-screen app: no widget chrome, gruvbox styling, one line per Row."
    def __init__(self, src, profile='standard'):
        self.st = InspectorState(src, profile)
        self._build()

    def _body_fragments(self):
        out = []
        for i, r in enumerate(self.st.rows):
            frags = r.frags
            if i == self.st.cursor:
                bar = [('class:bar', '▎')]
                frags = bar + [(f'class:cursor {s}'.strip(), t) for s, t in r.frags]
            else:
                frags = [(' ', ' ')] + list(r.frags)
            out += frags
            out.append(('', '\n'))
        if not out: out = [('class:more', '  (empty namespace)')]
        return out

    def _cursor_pos(self):
        from prompt_toolkit.data_structures import Point
        return Point(x=0, y=self.st.cursor)

    def _header_fragments(self):
        st = self.st
        n = sum(1 for r in st.rows if r.kind == 'node' and r.depth == 0)
        left = [('class:header', ' paar '), ('class:header.dim', ' inspector  ')]
        mid = [('class:header.dim', 'profile '), ('class:header', st.profile),
               ('class:header.dim', f'   {n} top-level ')]
        path = ''
        if st.cur is not None and getattr(st.cur, 'v', None) is not None and st.cur.kind == 'node':
            path = getattr(st.cur.v, 'path', '') or ''
        right = [('class:path', f' {path} ')] if path else []
        return left + mid + right

    def _build(self):
        from prompt_toolkit.application import Application
        from prompt_toolkit.key_binding import KeyBindings
        from prompt_toolkit.layout import Layout, HSplit, Window
        from prompt_toolkit.layout.controls import FormattedTextControl
        from prompt_toolkit.layout.dimension import Dimension
        from prompt_toolkit.layout.margins import ScrollbarMargin

        body = Window(
            content=FormattedTextControl(self._body_fragments, get_cursor_position=self._cursor_pos,
                                         focusable=True, show_cursor=False),
            style='class:body', wrap_lines=False, always_hide_cursor=True,
            right_margins=[ScrollbarMargin(display_arrows=False)])
        header = Window(FormattedTextControl(self._header_fragments), height=1, style='class:header')
        footer = Window(FormattedTextControl(lambda: _FOOTER), height=1, style='class:footer')
        layout = Layout(HSplit([header, body, footer]), focused_element=body)

        kb = KeyBindings()
        st = self.st

        def on(*keys):
            "Bind one handler to several keys; the handler takes no args and we invalidate after it."
            def deco(fn):
                def h(e): fn(); e.app.invalidate()
                for k in keys: kb.add(k)(h)
                return fn
            return deco

        kb.add('q')(lambda e: e.app.exit())
        kb.add('c-c')(lambda e: e.app.exit())
        on('j', 'down')(lambda: st.move(1))
        on('k', 'up')(lambda: st.move(-1))
        on('c-d')(lambda: st.move(10))
        on('c-u')(lambda: st.move(-10))
        on('g', 'home')(st.top)
        on('G', 'end')(st.bottom)
        on('l', 'right', 'enter', ' ')(st.toggle)
        on('h', 'left')(st.collapse)
        on('v')(st.toggle_grid)
        on('[')(lambda: st.page_grid(drow=-1))
        on(']')(lambda: st.page_grid(drow=1))
        on('{')(lambda: st.page_grid(dcol=-1))
        on('}')(lambda: st.page_grid(dcol=1))
        on('p')(lambda: st.cycle_profile(1))
        on('P')(lambda: st.cycle_profile(-1))
        on('r')(st.hard_refresh)

        self.app = Application(layout=layout, key_bindings=kb, style=GRUVBOX(),
                               full_screen=True, mouse_support=False)

    def run(self): return self.app.run()

In [8]:
#| export
def _live_ns():
    "The live IPython user namespace and its hidden set, or ({}, frozenset()) outside IPython."
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip: return ip.user_ns, frozenset(getattr(ip, 'user_ns_hidden', ()))
    except Exception: pass
    return {}, frozenset()

def inspector(ns=None, profile='standard'):
    "Launch the full-screen gruvbox inspector. ns=None reads the live IPython namespace; pass a dict to inspect it directly."
    if ns is None:
        ns, hidden = _live_ns()
        src = NsSource(ns, hidden)
    else:
        src = NsSource(ns)
    return InspectorTUI(src, profile).run()

In [9]:
#| hide
import math, numpy as np, pandas as pd
from paar.tui import NsSource, InspectorState, InspectorTUI, GRUVBOX

data = {'a': 1, 'b': [1, 2, {'deep': [10, 20]}], 'c': 'hello', 'pi': math.pi, 'flag': True, 'none': None}
arr = np.arange(12).reshape(3, 4)
df = pd.DataFrame({'x': range(5), 'y': list('abcde')})
ns = dict(data=data, arr=arr, df=df, big=list(range(500)))
src = NsSource(ns)

In [10]:
#| hide
# flatten + lazy tree expand + nested drill-in
st = InspectorState(src, 'standard')
assert len(st.rows) == 4               # data, arr, df, big (all 'data' category, shown flat)
st.cursor = next(i for i,r in enumerate(st.rows) if getattr(r.v,'name','')=='data')
st.toggle()
assert any(getattr(r.v,'name','')=="'b'" for r in st.rows if r.kind=='node')
st.cursor = next(i for i,r in enumerate(st.rows) if getattr(r.v,'name','')=="'b'")
st.toggle()
assert any(getattr(r.v,'name','')=='2' and r.v.is_container for r in st.rows if r.kind=='node')

In [11]:
#| hide
# grid render + column paging on an ndarray, then hide it
st = InspectorState(src, 'standard')
st.cursor = next(i for i,r in enumerate(st.rows) if getattr(r.v,'name','')=='arr')
st.toggle_grid()
assert any(r.kind=='gridline' for r in st.rows)
st.page_grid(dcol=1)
assert st.grids[('arr',)][1] > 0
st.toggle_grid()
assert not any(r.kind=='gridline' for r in st.rows)

In [12]:
#| hide
# load-more pagination: a 500-element list pages 100 children at a time
st = InspectorState(src, 'standard')
st.cursor = next(i for i,r in enumerate(st.rows) if getattr(r.v,'name','')=='big')
st.toggle()
kids = lambda: sum(1 for r in st.rows if r.kind=='node' and r.depth==1)
assert kids() == 100 and any(r.kind=='more' for r in st.rows)
st.cursor = next(i for i,r in enumerate(st.rows) if r.kind=='more')
st.toggle()
assert kids() == 200

In [13]:
#| hide
# every profile flattens, the gruvbox Style builds, and the app renders fragments
for p in ('minimal','standard','full'):
    InspectorState(src, p).refresh()
assert GRUVBOX() is not None
tui = InspectorTUI(src, 'standard')
assert tui._body_fragments() and tui._header_fragments()

In [14]:
#| hide
import nbdev; nbdev.nbdev_export()